In [13]:
from footnet_model import CustomDataLoader, LSTM_custom
import torch
from obtain_metrics import configure_data, compute_all_metrics
import pickle
from torch.utils.data import DataLoader
from scipy import stats
import cv2

#Loading the model:
model = LSTM_custom(input_size=4, hidden_size=32, num_layers=1, num_classes=1)
model.load_state_dict(torch.load("/Users/abhinavarora/Desktop/CadenceCV/models/footnet_lstm_best.pth", weights_only=True))
model.eval()

video_dir = '/Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4'
#Computing metrics from the video:
frame_by_frame_data = compute_all_metrics(video_dir)
#Loading mean and std from pickle file

with open("/Users/abhinavarora/Desktop/CadenceCV/models/scaler_means_and_dev.pkl", "rb") as file:
    data = pickle.load(file)

means = data["Means"]
stds = data["Stds"]

#Running data through the CustomDataLoader
all_features, all_masks, all_frames = configure_data(frame_by_frame_data, means, stds)
dataset = CustomDataLoader(all_features, all_masks)

batch_size = 32
#shuffle=False so batch i always maps to windows [i*batch_size, (i+1)*batch_size)
video_data = DataLoader(dataset, batch_size=batch_size, shuffle=False)

#Inference loop stores unmasked predictions (shape: batch_size x 40) so the
#vote loop can index per-window using i * batch_size
all_predictions = []
with torch.no_grad():
    for (ind, data) in enumerate(video_data):
        features, masks = data
        logits = model(features)
        predictions = (torch.sigmoid(logits) > 0.35).long()
        all_predictions.append(predictions)


video 1/1 (frame 1/300) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4: 1280x1024 1 person, 2226.7ms
video 1/1 (frame 2/300) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4: 1280x1024 2 persons, 2082.4ms
video 1/1 (frame 3/300) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4: 1280x1024 2 persons, 2119.1ms
video 1/1 (frame 4/300) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4: 1280x1024 1 person, 2041.9ms
video 1/1 (frame 5/300) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4: 1280x1024 1 person, 1967.0ms
video 1/1 (frame 6/300) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4: 1280x1024 (no detections), 1889.6ms
video 1/1 (frame 7/300) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4: 1280x1024 (no detections), 1880.5ms
video 1/1 (frame 8/300) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4: 1280x1024 1 person, 1880.5ms
video 1/1 (frame 9/300) /Users/abhinavarora/Desktop/CadenceCV/Videos/Video17.mp4: 1280x1024 1 p

In [10]:
print(all_frames.to(torch.int))
print(all_predictions)

tensor([[  0,   1,   2,  ...,  37,  38,  39],
        [  2,   3,   4,  ...,  39,  40,  41],
        [  4,   5,   6,  ...,  41,  42,  43],
        ...,
        [ -1,  -1,  -1,  ..., 247, 248, 249],
        [ -1,  -1,  -1,  ..., 247, 248, 249],
        [ -1,  -1,  -1,  ...,  -1, 248, 249]], dtype=torch.int32)
[tensor([1, 1, 1,  ..., 0, 0, 0]), tensor([1, 1, 1,  ..., 0, 0, 0]), tensor([1, 1, 0,  ..., 0, 0, 0]), tensor([1, 1, 1,  ..., 0, 0, 0]), tensor([1, 1, 1,  ..., 0, 0, 0]), tensor([1, 1, 1,  ..., 0, 0, 0]), tensor([1, 1, 1,  ..., 0, 0, 0]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1

In [ ]:
#Sliding window majority vote where each frame collects predictions from every window that covered it
#The predictions given by the LSTM are batched, meaning that each batch is of shape (32, 40) and not 
#(32, 40, 1) due to output.squeeze done in footnet_model. This corresponds to 32 chunks of size 40 frames (The resampling num)
#with a prediction attached to it. So, each batch must be iterated over where 32 chunks of size 40 are obtained.
#Instead of iterating over contiguous chunks, chunks in continguous batches are instead iterated over
frame_to_pred_dict = {}
for (i, batch_preds) in enumerate(all_predictions):
    #The first 40 frame window always will start at i * batch_size and the last 40 frame window will
    #be found at (i + 1) * batch_size
    window_start = i * batch_size
    #Last batch may not have 32 chunks
    batch_size_actual = batch_preds.shape[0]

    #Now this is iterating over the actual 40-sized framed windows
    for j in range(batch_size_actual):
        window_idx = window_start + j
        #Apply mask to get only real (non-padded) frames and predictions for this window
        window_mask = all_masks[window_idx]
        #applying the specific mask to a specific window in this current batch
        window_frames = all_frames[window_idx][window_mask]
        window_preds = batch_preds[j][window_mask]

        #Finally constructing the frame_to_pred_dict : {frame: predictions over each window}
        for (ind, frame) in enumerate(window_frames):
            frame_int = int(frame.item())
            if frame_int not in frame_to_pred_dict:
                frame_to_pred_dict[frame_int] = []
            frame_to_pred_dict[frame_int].append(window_preds[ind].item())

#Taking a majiority vote via mode
for frame in frame_to_pred_dict:
    frame_to_pred_dict[frame] = stats.mode(frame_to_pred_dict[frame], keepdims=True).mode[0]

print(frame_to_pred_dict)

#Detecting where strikefoot is actually happening 
strikefoot_frames = []
prev_pred = 0
for f in sorted(frame_to_pred_dict.keys()):
    curr_pred = frame_to_pred_dict[f]
    if curr_pred == 1 and prev_pred == 0:
        strikefoot_frames.append(f)
    prev_pred = curr_pred

cap = cv2.VideoCapture(video_dir)
for f in strikefoot_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(f))
    ret, img = cap.read()
    cv2.imshow("Strikefoot frame", img)
    cv2.waitKey(0)

cv2.destroyAllWindows()

{0: np.int64(1), 1: np.int64(1), 2: np.int64(1), 3: np.int64(1), 4: np.int64(1), 5: np.int64(1), 6: np.int64(1), 7: np.int64(1), 8: np.int64(1), 9: np.int64(1), 10: np.int64(1), 11: np.int64(1), 12: np.int64(1), 13: np.int64(1), 14: np.int64(1), 15: np.int64(1), 16: np.int64(1), 17: np.int64(1), 18: np.int64(1), 19: np.int64(1), 20: np.int64(0), 21: np.int64(0), 22: np.int64(0), 23: np.int64(0), 24: np.int64(0), 25: np.int64(1), 26: np.int64(1), 27: np.int64(1), 28: np.int64(1), 29: np.int64(1), 30: np.int64(1), 31: np.int64(1), 32: np.int64(1), 33: np.int64(1), 34: np.int64(0), 35: np.int64(0), 36: np.int64(0), 37: np.int64(0), 38: np.int64(0), 39: np.int64(0), 40: np.int64(1), 41: np.int64(1), 42: np.int64(1), 43: np.int64(1), 44: np.int64(1), 45: np.int64(1), 46: np.int64(1), 47: np.int64(1), 48: np.int64(1), 49: np.int64(1), 50: np.int64(0), 51: np.int64(0), 52: np.int64(0), 53: np.int64(0), 54: np.int64(0), 55: np.int64(0), 56: np.int64(0), 57: np.int64(0), 58: np.int64(0), 59: np